# **Data Analysis**

The goal of this file is to conduct data analysis for the Los Angeles Angels (LAA) and Los Angeles Dodgers (LAD) trade, home run, and score data.

In [1]:
import random
import pandas as pd
import plotly.express as px
from datetime import timedelta
from utils.data import DataProcessing
from utils.data_analysis import plot_corr_matrix

In [2]:
# Instantiate DataProcessing objects for LAA and LAD

laa_DP = DataProcessing(
    team = 'LAA',
    trade_path = "../data/raw/laa_kalshi_trade_data.parquet",
    score_path = "../data/raw/laa_score_data.parquet",
    homerun_path = "../data/raw/laa_homerun_data.parquet"
)

laa_trade_df = laa_DP.get_trade_df()
laa_score_df = laa_DP.get_score_df()
laa_homerun_df = laa_DP.get_homerun_df()

lad_DP = DataProcessing(
    team = 'LAD',
    trade_path = "../data/raw/lad_kalshi_trade_data.parquet",
    score_path = "../data/raw/lad_score_data.parquet",
    homerun_path = "../data/raw/lad_homerun_data.parquet"
)

lad_trade_df = lad_DP.get_trade_df()
lad_score_df = lad_DP.get_score_df()
lad_homerun_df = lad_DP.get_homerun_df()

## Graphical Analysis for LAA

In [3]:
# Merge LAA trade and home run frames

homerun_df_sorted = laa_homerun_df.copy().sort_values('time_pst').reset_index(drop=True)
trade_df_sorted = laa_trade_df.copy().sort_values('time_pst').reset_index(drop=True)

laa_hr_events = pd.merge_asof(
    homerun_df_sorted,
    trade_df_sorted,
    on = 'time_pst',
    by = 'ticker',
    direction = 'backward',
    suffixes = ('', '_at_hr'),
    tolerance = pd.Timedelta('3min')
)

laa_df = pd.merge(
    trade_df_sorted,
    laa_hr_events,
    how='outer'
)

laa_df['laa_homerun_dummy'] = laa_df['laa_homerun_dummy'].fillna(0).astype(int)
laa_df['homerun_dummy'] = laa_df['homerun_dummy'].fillna(0).astype(int)

laa_df = laa_df.dropna(subset=['yes_price_dollars']).reset_index(drop=True)

laa_df = laa_df[[
    'ticker', 'time_pst', 'yes_price_dollars',
    'laa_homerun_dummy', 'homerun_dummy'
]]

laa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420328 entries, 0 to 420327
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype                              
---  ------             --------------   -----                              
 0   ticker             420328 non-null  object                             
 1   time_pst           420328 non-null  datetime64[ns, America/Los_Angeles]
 2   yes_price_dollars  420328 non-null  float64                            
 3   laa_homerun_dummy  420328 non-null  int64                              
 4   homerun_dummy      420328 non-null  int64                              
dtypes: datetime64[ns, America/Los_Angeles](1), float64(1), int64(2), object(1)
memory usage: 16.0+ MB


In [4]:
unique_tickers = list(set(laa_df['ticker']))
random_ticker_subset = random.sample(unique_tickers, k=min(10, len(unique_tickers)))

laa_df['time_pst'] = pd.to_datetime(laa_df['time_pst'])
homerun_subset = laa_df[
    (laa_df['homerun_dummy'] == 1) & 
    (laa_df['ticker'].isin(random_ticker_subset))
].copy()

laa_df

,ticker,time_pst,yes_price_dollars,laa_homerun_dummy,homerun_dummy
0,KXMLBGAME-25APR16LAATEX-LAA,2025-04-16 10:13:32.448972-07:00,0.57,0,0
1,KXMLBGAME-25APR16LAATEX-LAA,2025-04-16 10:19:40.069928-07:00,0.57,0,0
2,KXMLBGAME-25APR16LAATEX-LAA,2025-04-16 10:23:44.258530-07:00,0.57,0,0
3,KXMLBGAME-25APR16LAATEX-LAA,2025-04-16 11:42:21.927767-07:00,0.57,0,0
4,KXMLBGAME-25APR16LAATEX-LAA,2025-04-16 11:46:16.656897-07:00,0.57,0,0
...,...,...,...,...,...
420323,KXMLBGAME-26MAY171607LADLAA-LAA,2026-05-17 15:37:11.675217-07:00,0.01,0,0
420324,KXMLBGAME-26MAY171607LADLAA-LAA,2026-05-17 15:37:35.703172-07:00,0.01,0,0
420325,KXMLBGAME-26MAY171607LADLAA-LAA,2026-05-17 15:37:49.479503-07:00,0.01,0,0
420326,KXMLBGAME-26MAY171607LADLAA-LAA,2026-05-17 15:39:10.673663-07:00,0.01,0,0


In [5]:
# Plot home runs 

laa_colors = ['#BA0021', '#C4CED4', '#003263']

unique_tickers = list(set(laa_df['ticker']))
random_ticker_subset = random.sample(unique_tickers, k=min(10, len(unique_tickers)))

laa_df['time_pst'] = pd.to_datetime(laa_df['time_pst'])
homerun_subset = laa_df[
    (laa_df['homerun_dummy'] == 1) & 
    (laa_df['ticker'].isin(random_ticker_subset))
].copy()

for idx, hr_event in homerun_subset.iterrows():
    hr_time = hr_event['time_pst']
    ticker = hr_event['ticker']
    is_laa = hr_event['laa_homerun_dummy'] == 1
    
    start_window = hr_time - timedelta(minutes=6)
    end_window = hr_time + timedelta(minutes=6)
    
    game_window = laa_df[
        (laa_df['ticker'] == ticker) & 
        (laa_df['time_pst'] >= start_window) & 
        (laa_df['time_pst'] <= end_window)
    ].copy()

    if game_window.empty:
        continue

    date_str = f"{ticker[12:15]} {ticker[15:17]}, 20{ticker[10:12]}"
    ticker_split_str = ticker[17:].split('-')[0].replace('LAA', '')
    opp_team = ''.join([char for char in ticker_split_str if not char.isdigit()])
    hr_type = "LAA" if is_laa else "Opponent"

    fig = px.line(game_window, x='time_pst', y='yes_price_dollars', color_discrete_sequence=[laa_colors[0]])

    fig.update_layout(
        template='plotly_dark',
        title={
            'text': f"<b>{hr_type} Home Run</b><br><span style='font-size:16px; color:{laa_colors[1]}'>LAA vs {opp_team} {date_str}</span>",
            'font': {'size': 20, 'color': laa_colors[1]},
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title='Time (PST)',
        yaxis_title='Price (LAA wins)',
        hovermode='x unified',
        paper_bgcolor='#111111', 
        plot_bgcolor='#111111'
    )

    label_text = "LAA HR" if is_laa else "Opponent HR"
    border_color = laa_colors[1] if is_laa else laa_colors[2]
    bg_color = 'rgba(186, 0, 33, 0.9)' if is_laa else 'rgba(0, 50, 99, 0.7)'

    fig.add_annotation(
        x=hr_time,
        y=hr_event['yes_price_dollars'],
        text=f"<b>{label_text}</b>",
        showarrow=True,
        arrowhead=2,
        arrowcolor='white', 
        ax=0,
        ay=60,         
        font=dict(color='white', size=11),
        bgcolor=bg_color,
        bordercolor=border_color,
        borderwidth=2,
        xanchor='center',
        yanchor='top' 
    )

    fig.show()

## Graphical Analysis for LAD

In [6]:
# Merge LAD trade and home run frames

homerun_df_sorted = lad_homerun_df.copy().sort_values('time_pst').reset_index(drop=True)
trade_df_sorted = lad_trade_df.copy().sort_values('time_pst').reset_index(drop=True)

lad_hr_events = pd.merge_asof(
    homerun_df_sorted,
    trade_df_sorted,
    on = 'time_pst',
    by = 'ticker',
    direction = 'backward',
    suffixes = ('', '_at_hr'),
    tolerance = pd.Timedelta('3min')
)

lad_df = pd.merge(
    trade_df_sorted,
    lad_hr_events,
    how='outer'
)

lad_df['lad_homerun_dummy'] = lad_df['lad_homerun_dummy'].fillna(0).astype(int)
lad_df['homerun_dummy'] = lad_df['homerun_dummy'].fillna(0).astype(int)

lad_df = lad_df.dropna(subset=['yes_price_dollars']).reset_index(drop=True)

lad_df = lad_df[[
    'ticker', 'time_pst', 'yes_price_dollars',
    'lad_homerun_dummy', 'homerun_dummy'
]]

lad_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1144010 entries, 0 to 1144009
Data columns (total 5 columns):
 #   Column             Non-Null Count    Dtype                              
---  ------             --------------    -----                              
 0   ticker             1144010 non-null  object                             
 1   time_pst           1144010 non-null  datetime64[ns, America/Los_Angeles]
 2   yes_price_dollars  1144010 non-null  float64                            
 3   lad_homerun_dummy  1144010 non-null  int64                              
 4   homerun_dummy      1144010 non-null  int64                              
dtypes: datetime64[ns, America/Los_Angeles](1), float64(1), int64(2), object(1)
memory usage: 43.6+ MB


In [7]:
# Plot home runs 

lad_colors = ['#EF3E42', '#A5ACAF', '#005A9C']

unique_tickers = list(set(lad_df['ticker']))
random_ticker_subset = random.sample(unique_tickers, k=min(10, len(unique_tickers)))

lad_df['time_pst'] = pd.to_datetime(lad_df['time_pst'])

homerun_subset = lad_df[
    (lad_df['homerun_dummy'] == 1) & 
    (lad_df['ticker'].isin(random_ticker_subset))
].copy()

for idx, hr_event in homerun_subset.iterrows():
    hr_time = hr_event['time_pst']
    ticker = hr_event['ticker']
    is_lad = hr_event['lad_homerun_dummy'] == 1
    
    start_window = hr_time - timedelta(minutes=6)
    end_window = hr_time + timedelta(minutes=6)
    
    game_window = lad_df[
        (lad_df['ticker'] == ticker) & 
        (lad_df['time_pst'] >= start_window) & 
        (lad_df['time_pst'] <= end_window)
    ].copy()

    if game_window.empty:
        continue

    date_str = f"{ticker[12:15]} {ticker[15:17]}, 20{ticker[10:12]}"
    ticker_split_str = ticker[17:].split('-')[0].replace('LAD', '')
    opp_team = ''.join([char for char in ticker_split_str if not char.isdigit()])
    hr_type = "LAD" if is_lad else "Opponent"

    fig = px.line(game_window, x='time_pst', y='yes_price_dollars', color_discrete_sequence=[lad_colors[2]])

    fig.update_layout(
        template='plotly_dark',
        title={
            'text': f"<b>{hr_type} Home Run</b><br><span style='font-size:16px; color:{lad_colors[1]}'>LAD vs {opp_team} {date_str}</span>",
            'font': {'size': 20, 'color': 'white'},
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title='Time (PST)',
        yaxis_title='Price (LAD wins)',
        hovermode='x unified',
        paper_bgcolor='#0a0a0a', 
        plot_bgcolor='#0a0a0a'
    )

    label_text = "LAD HR" if is_lad else "Opponent HR"
    border_color = lad_colors[0] if is_lad else lad_colors[1]
    bg_color = 'rgba(0, 90, 156, 0.9)' if is_lad else 'rgba(60, 60, 60, 0.8)'

    fig.add_annotation(
        x=hr_time,
        y=hr_event['yes_price_dollars'],
        text=f"<b>{label_text}</b>",
        showarrow=True,
        arrowhead=2,
        arrowcolor='white',
        ax=0,
        ay=60, 
        font=dict(color='white', size=11),
        bgcolor=bg_color,
        bordercolor=border_color,
        borderwidth=2,
        xanchor='center',
        yanchor='top' 
    )

    fig.show()

### Observations

- Generally, home runs seem to occur near large price movements, but they do not always align with what would be expected. The theory is that a home run occurs prior to a large directional price movement:

    - If the opponent hits a home run, then the price goes down.

    - If LAA/LAD hits a home run, then the price goes up.

- Also, home runs do not always lead to large price movements.


### Model Implications

- Generic Price Changes

    - Just look at the price for the next trade.

- Forward Binned Mean Price Changes

    - Define a window parameter $w$ that determines how many trades to skip after the home run event.

    - Define a bin parameter $b$ that determines how many trades to include in the forward bin.
    
    - The forward bin includes $b$ trades starting from $w$ trades after the home run: indices $[i + w, i + w + b - 1]$
    
    - The forward binned price change is defined as: $$p_{f}(w,b) = \bar{p}_{forward}(w,b) - p_{hr}$$

    - The percent forward price change is defined as: $$\tilde{p}_{f}(w,b) = \frac{\bar{p}_{forward}(w,b) - p_{hr}}{p_{hr}}$$

        - Where $\bar{p}_{forward}(w,b)$ is the average price in the forward bin containing $b$ trades, and $p_{hr}$ is the price at the home run event.

- Double Binned Mean Price Changes

    - Assume the home run data (from `mlbstatsapi`) is slightly delayed.

    - Define a window parameter $w$ that determines how many trades to skip on either side of the recorded home run event.

    - Define a bin parameter $b$ that determines how many trades to include in each bin.
    
    - The "before" bin includes $b$ trades starting from $w$ trades before the home run: indices $[i - w - b + 1, i - w]$
    
    - The "after" bin includes $b$ trades starting from $w$ trades after the home run: indices $[i + w, i + w + b - 1]$
    
    - The double binned price change is defined as: $$p_{d}(w,b) = \bar{p}_{after}(w,b) - \bar{p}_{before}(w,b)$$

    - The percent price change is defined as: $$\tilde{p}_{d}(w,b) = \frac{\bar{p}_{after}(w,b) - \bar{p}_{before}(w,b)}{\bar{p}_{before}(w,b)}$$

        - Where $\bar{p}_{after}(w,b)$ is the average price in the "after" bin and $\bar{p}_{before}(w,b)$ is the average price in the "before" bin.
        
        - Both bins contain exactly $b$ trades each, positioned $w$ trades away from the home run event.

    - Additionally, the double binned mean price can be defined as:$$\bar{p}_{before}(w,b)$$

- Price changes were chosen for this analysis; within a bounded market, first differences provide a clearer representation of market sentiment than percentage changes.

## Statistical Analysis for LAA

In [8]:
# Load in feature engineered data frames

laa_generic_df = pd.read_parquet("../data/processed/laa_generic_data.parquet")
laa_forward_df = pd.read_parquet("../data/processed/laa_forward_data.parquet")
laa_double_df = pd.read_parquet("../data/processed/laa_double_data.parquet")

### Generic Data Frame Analysis

In [9]:
# Define features and targets

generic_features = [
    'yes_price_dollars', 'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

generic_targets = ['g_px_chg']

# Look at a subset of the features, and drop any price change data if missing
laa_generic_df = laa_generic_df[generic_targets + generic_features].dropna(subset=['g_px_chg'])

laa_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 339 entries, 0 to 362
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_px_chg           339 non-null    float64
 1   yes_price_dollars  339 non-null    float64
 2   laa_homerun_dummy  339 non-null    int64  
 3   runners_on_base    339 non-null    float64
 4   inning             339 non-null    float64
 5   laa_score          339 non-null    float64
 6   opp_score          339 non-null    float64
 7   score_delta        339 non-null    float64
 8   score_delta_abs    339 non-null    float64
dtypes: float64(8), int64(1)
memory usage: 26.5 KB


In [10]:
# General descriptive statistics

print("\n\nLAA Home Runs:")
laa_hr_df = laa_generic_df[laa_generic_df['laa_homerun_dummy'] == 1]
print(round(laa_hr_df[generic_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = laa_generic_df[laa_generic_df['laa_homerun_dummy'] != 1]
print(round(opp_hr_df[generic_targets].describe(), 2))



LAA Home Runs:
       g_px_chg
count    180.00
mean       0.05
std        0.08
min       -0.22
25%        0.00
50%        0.04
75%        0.08
max        0.47

Opponent Home Runs:
       g_px_chg
count    159.00
mean      -0.08
std        0.10
min       -0.66
25%       -0.12
50%       -0.06
75%       -0.01
max        0.16


In [11]:
# Correlation matrix

fig = plot_corr_matrix(
    df = laa_generic_df,
    df_name = 'Generic',
    feature_cols = generic_features,
    target_cols = generic_targets,
    team = 'LAA',
    team_colors = laa_colors
)

fig.show()

### Forward Data Frame Analysis

In [12]:
# Define features and targets

forward_features = [
    'yes_price_dollars', 'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

forward_targets = [
    'f_px_chg_w1_b2', 'f_px_chg_w2_b2', 'f_px_chg_w3_b2', 
    'f_px_chg_w4_b2', 'f_px_chg_w5_b2', 'f_px_chg_w6_b2',
    'f_px_chg_w1_b4', 'f_px_chg_w2_b4', 'f_px_chg_w3_b4', 
    'f_px_chg_w4_b4', 'f_px_chg_w5_b4', 'f_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
laa_forward_df = laa_forward_df[forward_targets + forward_features].dropna(subset=forward_targets)

laa_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 321 entries, 0 to 362
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   f_px_chg_w1_b2     321 non-null    float64
 1   f_px_chg_w2_b2     321 non-null    float64
 2   f_px_chg_w3_b2     321 non-null    float64
 3   f_px_chg_w4_b2     321 non-null    float64
 4   f_px_chg_w5_b2     321 non-null    float64
 5   f_px_chg_w6_b2     321 non-null    float64
 6   f_px_chg_w1_b4     321 non-null    float64
 7   f_px_chg_w2_b4     321 non-null    float64
 8   f_px_chg_w3_b4     321 non-null    float64
 9   f_px_chg_w4_b4     321 non-null    float64
 10  f_px_chg_w5_b4     321 non-null    float64
 11  f_px_chg_w6_b4     321 non-null    float64
 12  yes_price_dollars  321 non-null    float64
 13  laa_homerun_dummy  321 non-null    int64  
 14  runners_on_base    321 non-null    float64
 15  inning             321 non-null    float64
 16  laa_score          321 non-null

In [13]:
# General descriptive statistics

print("\n\nLAA Home Runs:")
laa_hr_df = laa_forward_df[laa_forward_df['laa_homerun_dummy'] == 1]
print(round(laa_hr_df[forward_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = laa_forward_df[laa_forward_df['laa_homerun_dummy'] != 1]
print(round(opp_hr_df[forward_targets].describe(), 2))



LAA Home Runs:
       f_px_chg_w1_b2  f_px_chg_w2_b2  f_px_chg_w3_b2  f_px_chg_w4_b2  \
count          167.00          167.00          167.00          167.00   
mean             0.06            0.07            0.07            0.07   
std              0.08            0.09            0.09            0.09   
min             -0.22           -0.22           -0.22           -0.24   
25%              0.01            0.02            0.01            0.01   
50%              0.05            0.06            0.06            0.06   
75%              0.08            0.10            0.10            0.10   
max              0.47            0.47            0.48            0.49   

       f_px_chg_w5_b2  f_px_chg_w6_b2  f_px_chg_w1_b4  f_px_chg_w2_b4  \
count          167.00          167.00          167.00          167.00   
mean             0.07            0.07            0.06            0.07   
std              0.09            0.09            0.08            0.09   
min             -0.24           -

In [14]:
# Correlation matrix

fig = plot_corr_matrix(
    df = laa_forward_df,
    df_name = 'Forward Binned Mean',
    feature_cols = forward_features,
    target_cols = forward_targets,
    team = 'LAA',
    team_colors = laa_colors
)

fig.show()

### Double Data Frame Analysis

In [15]:
# Define features and targets

double_features = [
    'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

double_targets = [
    'd_px_chg_w1_b2', 'd_px_chg_w2_b2', 'd_px_chg_w3_b2', 
    'd_px_chg_w4_b2', 'd_px_chg_w5_b2', 'd_px_chg_w6_b2',
    'd_px_chg_w1_b4', 'd_px_chg_w2_b4', 'd_px_chg_w3_b4', 
    'd_px_chg_w4_b4', 'd_px_chg_w5_b4', 'd_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
laa_double_df = laa_double_df[double_targets + double_features].dropna(subset=double_targets)

laa_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 304 entries, 0 to 362
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   d_px_chg_w1_b2     304 non-null    float64
 1   d_px_chg_w2_b2     304 non-null    float64
 2   d_px_chg_w3_b2     304 non-null    float64
 3   d_px_chg_w4_b2     304 non-null    float64
 4   d_px_chg_w5_b2     304 non-null    float64
 5   d_px_chg_w6_b2     304 non-null    float64
 6   d_px_chg_w1_b4     304 non-null    float64
 7   d_px_chg_w2_b4     304 non-null    float64
 8   d_px_chg_w3_b4     304 non-null    float64
 9   d_px_chg_w4_b4     304 non-null    float64
 10  d_px_chg_w5_b4     304 non-null    float64
 11  d_px_chg_w6_b4     304 non-null    float64
 12  laa_homerun_dummy  304 non-null    int64  
 13  runners_on_base    304 non-null    float64
 14  inning             304 non-null    float64
 15  laa_score          304 non-null    float64
 16  opp_score          304 non-null

In [16]:
# General descriptive statistics

print("\n\nLAA Home Runs:")
laa_hr_df = laa_double_df[laa_double_df['laa_homerun_dummy'] == 1]
print(round(laa_hr_df[double_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = laa_double_df[laa_double_df['laa_homerun_dummy'] != 1]
print(round(opp_hr_df[double_targets].describe(), 2))



LAA Home Runs:
       d_px_chg_w1_b2  d_px_chg_w2_b2  d_px_chg_w3_b2  d_px_chg_w4_b2  \
count          158.00          158.00          158.00          158.00   
mean             0.07            0.10            0.11            0.12   
std              0.08            0.08            0.09            0.10   
min             -0.17           -0.11           -0.11           -0.10   
25%              0.02            0.05            0.05            0.05   
50%              0.06            0.08            0.10            0.10   
75%              0.11            0.14            0.16            0.17   
max              0.47            0.37            0.40            0.62   

       d_px_chg_w5_b2  d_px_chg_w6_b2  d_px_chg_w1_b4  d_px_chg_w2_b4  \
count          158.00          158.00          158.00          158.00   
mean             0.13            0.13            0.09            0.11   
std              0.11            0.11            0.07            0.08   
min             -0.08           -

In [17]:
# Correlation matrix

fig = plot_corr_matrix(
    df = laa_double_df,
    df_name = 'Double Binned Mean',
    feature_cols = double_features,
    target_cols = double_targets,
    team = 'LAA',
    team_colors = laa_colors
)

fig.show()

## Statistical Analysis for LAD

In [18]:
# Load in feature engineered data frames

lad_generic_df = pd.read_parquet("../data/processed/lad_generic_data.parquet")
lad_forward_df = pd.read_parquet("../data/processed/lad_forward_data.parquet")
lad_double_df = pd.read_parquet("../data/processed/lad_double_data.parquet")

### Generic Data Frame Analysis

In [19]:
# Define features and targets

generic_features = [
    'yes_price_dollars', 'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

generic_targets = ['g_px_chg']

# Look at a subset of the features, and drop any price change data if missing
lad_generic_df = lad_generic_df[generic_targets + generic_features].dropna(subset=['g_px_chg'])

lad_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 393 entries, 0 to 402
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_px_chg           393 non-null    float64
 1   yes_price_dollars  393 non-null    float64
 2   lad_homerun_dummy  393 non-null    int64  
 3   runners_on_base    393 non-null    float64
 4   inning             393 non-null    float64
 5   lad_score          393 non-null    float64
 6   opp_score          393 non-null    float64
 7   score_delta        393 non-null    float64
 8   score_delta_abs    393 non-null    float64
dtypes: float64(8), int64(1)
memory usage: 30.7 KB


In [20]:
# General descriptive statistics

print("\n\nLAD Home Runs:")
lad_hr_df = lad_generic_df[lad_generic_df['lad_homerun_dummy'] == 1]
print(round(lad_hr_df[generic_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = lad_generic_df[lad_generic_df['lad_homerun_dummy'] != 1]
print(round(opp_hr_df[generic_targets].describe(), 2))



LAD Home Runs:
       g_px_chg
count    230.00
mean       0.03
std        0.06
min       -0.15
25%        0.00
50%        0.01
75%        0.05
max        0.35

Opponent Home Runs:
       g_px_chg
count    163.00
mean      -0.05
std        0.12
min       -0.52
25%       -0.09
50%       -0.02
75%        0.00
max        0.65


In [21]:
# Correlation matrix

fig = plot_corr_matrix(
    df = lad_generic_df,
    df_name = 'Generic',
    feature_cols = generic_features,
    target_cols = generic_targets,
    team = 'LAD',
    team_colors = lad_colors
)

fig.show()

### Forward Data Frame Analysis

In [22]:
# Define features and targets

forward_features = [
    'yes_price_dollars', 'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

forward_targets = [
    'f_px_chg_w1_b2', 'f_px_chg_w2_b2', 'f_px_chg_w3_b2', 
    'f_px_chg_w4_b2', 'f_px_chg_w5_b2', 'f_px_chg_w6_b2',
    'f_px_chg_w1_b4', 'f_px_chg_w2_b4', 'f_px_chg_w3_b4', 
    'f_px_chg_w4_b4', 'f_px_chg_w5_b4', 'f_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
lad_forward_df = lad_forward_df[forward_targets + forward_features].dropna(subset=forward_targets)

lad_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 373 entries, 0 to 402
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   f_px_chg_w1_b2     373 non-null    float64
 1   f_px_chg_w2_b2     373 non-null    float64
 2   f_px_chg_w3_b2     373 non-null    float64
 3   f_px_chg_w4_b2     373 non-null    float64
 4   f_px_chg_w5_b2     373 non-null    float64
 5   f_px_chg_w6_b2     373 non-null    float64
 6   f_px_chg_w1_b4     373 non-null    float64
 7   f_px_chg_w2_b4     373 non-null    float64
 8   f_px_chg_w3_b4     373 non-null    float64
 9   f_px_chg_w4_b4     373 non-null    float64
 10  f_px_chg_w5_b4     373 non-null    float64
 11  f_px_chg_w6_b4     373 non-null    float64
 12  yes_price_dollars  373 non-null    float64
 13  lad_homerun_dummy  373 non-null    int64  
 14  runners_on_base    373 non-null    float64
 15  inning             373 non-null    float64
 16  lad_score          373 non-null

In [23]:
# General descriptive statistics

print("\n\nLAD Home Runs:")
lad_hr_df = lad_forward_df[lad_forward_df['lad_homerun_dummy'] == 1]
print(round(lad_hr_df[forward_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = lad_forward_df[lad_forward_df['lad_homerun_dummy'] != 1]
print(round(opp_hr_df[forward_targets].describe(), 2))



LAD Home Runs:
       f_px_chg_w1_b2  f_px_chg_w2_b2  f_px_chg_w3_b2  f_px_chg_w4_b2  \
count          221.00          221.00          221.00          221.00   
mean             0.04            0.04            0.04            0.05   
std              0.07            0.07            0.07            0.07   
min             -0.11           -0.09           -0.09           -0.08   
25%              0.00            0.00            0.00            0.00   
50%              0.01            0.02            0.03            0.03   
75%              0.06            0.07            0.07            0.07   
max              0.39            0.43            0.42            0.43   

       f_px_chg_w5_b2  f_px_chg_w6_b2  f_px_chg_w1_b4  f_px_chg_w2_b4  \
count          221.00          221.00          221.00          221.00   
mean             0.05            0.05            0.04            0.04   
std              0.07            0.07            0.07            0.07   
min             -0.08           -

In [24]:
# Correlation matrix

fig = plot_corr_matrix(
    df = lad_forward_df,
    df_name = 'Forward Binned Mean',
    feature_cols = forward_features,
    target_cols = forward_targets,
    team = 'LAD',
    team_colors = lad_colors
)

fig.show()

### Double Data Frame Analysis

In [25]:
# Define features and targets

double_features = [
    'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

double_targets = [
    'd_px_chg_w1_b2', 'd_px_chg_w2_b2', 'd_px_chg_w3_b2', 
    'd_px_chg_w4_b2', 'd_px_chg_w5_b2', 'd_px_chg_w6_b2',
    'd_px_chg_w1_b4', 'd_px_chg_w2_b4', 'd_px_chg_w3_b4', 
    'd_px_chg_w4_b4', 'd_px_chg_w5_b4', 'd_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
lad_double_df = lad_double_df[double_targets + double_features].dropna(subset=double_targets)

lad_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 361 entries, 0 to 402
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   d_px_chg_w1_b2     361 non-null    float64
 1   d_px_chg_w2_b2     361 non-null    float64
 2   d_px_chg_w3_b2     361 non-null    float64
 3   d_px_chg_w4_b2     361 non-null    float64
 4   d_px_chg_w5_b2     361 non-null    float64
 5   d_px_chg_w6_b2     361 non-null    float64
 6   d_px_chg_w1_b4     361 non-null    float64
 7   d_px_chg_w2_b4     361 non-null    float64
 8   d_px_chg_w3_b4     361 non-null    float64
 9   d_px_chg_w4_b4     361 non-null    float64
 10  d_px_chg_w5_b4     361 non-null    float64
 11  d_px_chg_w6_b4     361 non-null    float64
 12  lad_homerun_dummy  361 non-null    int64  
 13  runners_on_base    361 non-null    float64
 14  inning             361 non-null    float64
 15  lad_score          361 non-null    float64
 16  opp_score          361 non-null

In [26]:
# General descriptive statistics

print("\n\nLAD Home Runs:")
lad_hr_df = lad_double_df[lad_double_df['lad_homerun_dummy'] == 1]
print(round(lad_hr_df[double_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = lad_double_df[lad_double_df['lad_homerun_dummy'] != 1]
print(round(opp_hr_df[double_targets].describe(), 2))



LAD Home Runs:
       d_px_chg_w1_b2  d_px_chg_w2_b2  d_px_chg_w3_b2  d_px_chg_w4_b2  \
count          215.00          215.00          215.00          215.00   
mean             0.04            0.06            0.07            0.07   
std              0.06            0.07            0.07            0.07   
min             -0.07           -0.15           -0.10           -0.03   
25%              0.00            0.01            0.01            0.02   
50%              0.03            0.04            0.06            0.06   
75%              0.06            0.09            0.11            0.12   
max              0.39            0.42            0.39            0.41   

       d_px_chg_w5_b2  d_px_chg_w6_b2  d_px_chg_w1_b4  d_px_chg_w2_b4  \
count          215.00          215.00          215.00          215.00   
mean             0.08            0.08            0.05            0.07   
std              0.08            0.08            0.06            0.07   
min             -0.05           -

In [27]:
# Correlation matrix

fig = plot_corr_matrix(
    df = lad_double_df,
    df_name = 'Double Binned Mean',
    feature_cols = double_features,
    target_cols = double_targets,
    team = 'LAD',
    team_colors = lad_colors
)

fig.show()